# One-off: move the first silver table into the new silver tables

**Temporary. Delete this notebook and its job task after it has run once.**

The first version of silver kept everything in one table, `silver_pdfs` (parsed output, Markdown, and a map of parts). The new design (see `silver_pdfs.ipynb`) uses `silver_parsed`, `silver_pages` and `silver_log`. This notebook moves the existing work over **without calling the parser or the model again**, so nothing is paid for twice:

| From `silver_pdfs` | To | How many |
| --- | --- | --- |
| PDFs with a complete parse | `silver_parsed` | 304 |
| PDFs with page errors (rate limit) or no parse at all | `silver_log` (parse, attempt 1, retryable unless over 500 pages or conversion failed) | 55 retryable + 9 not retryable |
| The Markdown of every converted PDF, cut into pages at the `<!-- page N -->` markers | `silver_pages`, with the **real PDF page number** | about 46,600 |

The old Markdown used page numbers that count only the pages with content (1, 2, 3, ...). The new tables use the real page number (the parser's page id plus 1), so each old number is mapped through the sorted page ids of the PDF. Pages with an empty answer are left out, so they are converted again.

It can run again safely: every insert skips what is already there. The old table is not changed.

In [ ]:
dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("schema", "finance_bench_rag")
catalog = dbutils.widgets.get("catalog")
schema = dbutils.widgets.get("schema")

legacy = f"{catalog}.{schema}.silver_pdfs"
parsed_tbl = f"{catalog}.{schema}.silver_parsed"
pages_tbl = f"{catalog}.{schema}.silver_pages"
log_tbl = f"{catalog}.{schema}.silver_log"

# Same definitions as silver_pdfs.ipynb
spark.sql(f"CREATE TABLE IF NOT EXISTS {parsed_tbl} (path STRING, parsed VARIANT, pdf_pages INT, pages INT, parsed_at TIMESTAMP)")
spark.sql(f"CREATE TABLE IF NOT EXISTS {pages_tbl} (path STRING, page_no INT, markdown STRING, converted_at TIMESTAMP)")
spark.sql(f"""CREATE TABLE IF NOT EXISTS {log_tbl} (path STRING, stage STRING, page_no INT, attempt INT, retryable BOOLEAN,
                                                    reason STRING, pdf_pages INT, logged_at TIMESTAMP)""")

## 1. Complete parses to `silver_parsed`

A PDF is complete when it has elements and no `error_status`.

In [ ]:
spark.sql(f"""
INSERT INTO {parsed_tbl}
SELECT l.path, l.parsed, array_size(variant_get(l.parsed, '$.document.pages', 'ARRAY<VARIANT>')), pg.n, current_timestamp()
FROM {legacy} l
JOIN (SELECT path, count(DISTINCT coalesce(e.value:bbox[0].page_id::INT, 0)) AS n
      FROM {legacy}, LATERAL variant_explode(parsed:document:elements) e GROUP BY path) pg ON pg.path = l.path
WHERE coalesce(array_size(variant_get(l.parsed, '$.error_status', 'ARRAY<VARIANT>')), 0) = 0
  AND l.path NOT IN (SELECT path FROM {parsed_tbl})
""")
print(spark.table(parsed_tbl).count(), "rows in silver_parsed")

## 2. Incomplete parses to `silver_log`

PDFs with page errors are retryable. PDFs with no elements are retryable unless the parser said the PDF is over 500 pages or the conversion failed.

In [ ]:
spark.sql(f"""
INSERT INTO {log_tbl}
SELECT path, 'parse', CAST(NULL AS INT), 1,
       CASE WHEN n_el = 0 THEN NOT (msg RLIKE 'exceeds the limit|conversion failed') ELSE true END AS retryable,
       left(CASE WHEN n_el = 0 THEN coalesce(msg, 'no elements') ELSE concat(n_err, ' pages failed: ', msg) END, 500) AS reason,
       pdf_pages, current_timestamp()
FROM (
  SELECT path,
         array_size(variant_get(parsed, '$.document.elements', 'ARRAY<VARIANT>')) AS n_el,
         coalesce(array_size(variant_get(parsed, '$.error_status', 'ARRAY<VARIANT>')), 0) AS n_err,
         CAST(parsed:error_status[0].error_message AS STRING) AS msg,
         array_size(variant_get(parsed, '$.document.pages', 'ARRAY<VARIANT>')) AS pdf_pages
  FROM {legacy}
) x
WHERE (n_el = 0 OR n_err > 0)
  AND path NOT IN (SELECT path FROM {log_tbl} WHERE stage = 'parse')
""")
display(spark.table(log_tbl).groupBy("retryable").count())

## 3. The Markdown to `silver_pages`

Cuts each document's Markdown at its `<!-- page N -->` markers, removes the `---` that joined the pages, and maps the old page number to the real PDF page number.

In [ ]:
spark.sql(f"""
INSERT INTO {pages_tbl}
SELECT src.path, src.page_no, src.markdown, current_timestamp() FROM (
  WITH ids AS (
    SELECT path, array_sort(collect_set(coalesce(e.value:bbox[0].page_id::INT, 0))) AS page_ids
    FROM {legacy}, LATERAL variant_explode(parsed:document:elements) e WHERE markdown IS NOT NULL GROUP BY path),
  m AS (
    SELECT path, regexp_extract_all(markdown, '<!-- page ([0-9]+) -->', 1) AS nums, split(markdown, '<!-- page [0-9]+ -->') AS segs
    FROM {legacy} WHERE markdown IS NOT NULL)
  SELECT m.path AS path,
         element_at(ids.page_ids, cast(n AS INT)) + 1 AS page_no,
         regexp_replace(regexp_replace(m.segs[pos + 1], '(\\\\s*\\\\n---\\\\s*)+$', ''), '^\\\\s+|\\\\s+$', '') AS markdown
  FROM m JOIN ids ON ids.path = m.path LATERAL VIEW posexplode(m.nums) t AS pos, n
) src
WHERE length(src.markdown) > 0
  AND NOT EXISTS (SELECT 1 FROM {pages_tbl} g WHERE g.path = src.path AND g.page_no = src.page_no)
""")
print(spark.table(pages_tbl).count(), "rows in silver_pages")

## 4. Check

The numbers should be: 304 parsed, 64 logged (55 + 9), and `silver_pages` about 46,600 (the old marker total of 46,617 minus the empty pages).

In [ ]:
print("silver_parsed:", spark.table(parsed_tbl).count())
print("silver_log   :", spark.table(log_tbl).count())
print("silver_pages :", spark.table(pages_tbl).count())
print("old markers  :", spark.sql(f"SELECT sum(size(regexp_extract_all(markdown, '<!-- page ([0-9]+) -->', 1))) FROM {legacy} WHERE markdown IS NOT NULL").first()[0])
print("duplicate pages:", spark.sql(f"SELECT count(*) FROM (SELECT path, page_no FROM {pages_tbl} GROUP BY path, page_no HAVING count(*) > 1)").first()[0])
print("PDFs with no record at all:", spark.sql(f"SELECT count(*) FROM {legacy} WHERE path NOT IN (SELECT path FROM {parsed_tbl}) AND path NOT IN (SELECT path FROM {log_tbl})").first()[0])